# 04 — Advanced Bearing

The previous notebooks covered the fundamentals: what bearing is, how to compute it, and how to apply it to filtering and visualization.

This notebook goes further. The problems here arise at longer ranges, across multi-leg paths, and in scenarios where a single bearing is not enough:

- **Destination point** — given a start, a bearing, and a distance, where do you end up?
- **Midpoint bearing** — what is the heading at the halfway point of a long route?
- **Final bearing** — how does the heading at arrival differ from the heading at departure?
- **Great-circle path** — how do you build the actual arc between two points and sample it?
- **Bearing over time** — if a target is moving, how does the required bearing change?

These are the tools for trajectory planning, interception geometry, and anything that happens across distances where the Earth's curvature is no longer negligible.

In [ ]:
import math
import json
from pathlib import Path

# ── Core functions carried forward ──────────────────────────────────────────

def compute_bearing(p1, p2):
    """Initial bearing from p1 to p2. Inputs: [lon, lat]. Output: [0, 360)."""
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    d_lon = lon2 - lon1
    x = math.sin(d_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(d_lon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def haversine_km(p1, p2):
    """Great-circle distance in km. Inputs: [lon, lat]."""
    R = 6371.0
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def destination_point(origin, bearing_deg, distance_km):
    """Point reached by travelling distance_km from origin at bearing_deg."""
    R = 6371.0
    d   = distance_km / R
    brg = math.radians(bearing_deg)
    lat1 = math.radians(origin[1])
    lon1 = math.radians(origin[0])
    lat2 = math.asin(math.sin(lat1)*math.cos(d) + math.cos(lat1)*math.sin(d)*math.cos(brg))
    lon2 = lon1 + math.atan2(math.sin(brg)*math.sin(d)*math.cos(lat1),
                              math.cos(d) - math.sin(lat1)*math.sin(lat2))
    return [math.degrees(lon2), math.degrees(lat2)]

with open(Path("data/bearing_examples.json")) as f:
    examples = json.load(f)

print("Ready.")

## 1. Destination Point — The Direct Problem

`compute_bearing` solves the **inverse problem**: given two points, find the direction.

`destination_point` solves the **direct problem**: given a starting point, a bearing, and a distance, find the endpoint.

Both problems have exact spherical solutions. The direct problem is what you use to:
- project a missile's position after travelling a given distance on a given heading
- construct range rings (points at a fixed distance in all directions)
- build a flight path step by step

`destination_point` was introduced in the previous notebook. Here we examine it more carefully and verify it round-trips correctly with `compute_bearing` and `haversine_km`.

In [ ]:
origin = [-98.49, 33.91]  # Wichita Falls

# Round-trip check: project to a destination, then verify distance and bearing back
test_cases = [
    ("Due North",     0,   500),
    ("Due East",     90,   500),
    ("Due South",   180,   500),
    ("Due West",    270,   500),
    ("Northeast",    45,  1000),
    ("NNW",         337,   250),
]

print(f"{'Direction':<14} {'Brg':>6}  {'Dist':>8}  {'Recovered brg':>15}  {'Recovered dist':>16}  {'OK?':>5}")
print("-" * 72)
for label, brg, dist_km in test_cases:
    dest          = destination_point(origin, brg, dist_km)
    recovered_brg = compute_bearing(origin, dest)
    recovered_km  = haversine_km(origin, dest)
    brg_ok  = abs(recovered_brg - brg)   < 0.01
    dist_ok = abs(recovered_km  - dist_km) < 0.01
    ok = "✓" if brg_ok and dist_ok else "✗"
    print(f"{label:<14} {brg:>5}°  {dist_km:>6} km  {recovered_brg:>14.4f}°  {recovered_km:>14.4f} km  {ok:>5}")

Every row recovers both the original bearing and the original distance to sub-millimeter precision. The two functions are exact inverses of each other on a sphere.

## 2. Final Bearing

`compute_bearing(p1, p2)` returns the **initial bearing** — the heading at p1. The **final bearing** is the heading when you *arrive* at p2, looking back along the same great-circle arc.

On a flat surface these are always `180°` apart. On a sphere they are not.

The final bearing is simply the reverse initial bearing flipped by 180°: compute the bearing from p2 to p1, then add 180°.

In [ ]:
def final_bearing(p1, p2):
    """
    The bearing at which you arrive at p2 when travelling from p1 along a great circle.
    Equal to the reverse bearing (p2→p1) rotated 180°.
    """
    return (compute_bearing(p2, p1) + 180) % 360


# Compare initial vs. final bearing across routes of increasing length
wf = [-98.49, 33.91]

routes = [
    ("WF → Altus AFB (~90 km)",      [-99.27, 34.66]),
    ("WF → OKC (~220 km)",           [-97.52, 35.47]),
    ("WF → Chicago (~1400 km)",      [-87.63, 41.88]),
    ("WF → New York (~2400 km)",     [-74.01, 40.71]),
    ("WF → London (~8800 km)",       [ -0.13, 51.51]),
]

print(f"{'Route':<30} {'Initial':>10}  {'Final':>10}  {'Diff from 180°':>16}")
print("-" * 72)
for label, dest in routes:
    init  = compute_bearing(wf, dest)
    final = final_bearing(wf, dest)
    diff  = abs((final - init + 180) % 360 - 180)   # deviation from exactly 180° apart
    print(f"{label:<30} {init:>9.2f}°  {final:>9.2f}°  {diff:>+15.2f}°")

Short routes barely deviate from 180°. Long intercontinental routes diverge significantly. The final bearing to London is not simply "initial bearing + 180°" — the arc has curved enough that arrival heading and departure heading tell genuinely different stories.

## 3. Midpoint of a Great Circle

The midpoint of a great-circle arc is not simply the average of the two endpoints' coordinates. The correct midpoint lies on the arc itself.

The spherical midpoint formula computes it directly from the Cartesian vectors of the two endpoints.

In [ ]:
def great_circle_midpoint(p1, p2):
    """
    Spherical midpoint of the great-circle arc from p1 to p2.
    Inputs: [lon, lat] in decimal degrees.
    Returns: [lon, lat]
    """
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])

    # Convert to Cartesian unit vectors
    bx = math.cos(lat2) * math.cos(lon2 - lon1)
    by = math.cos(lat2) * math.sin(lon2 - lon1)

    mid_lat = math.atan2(
        math.sin(lat1) + math.sin(lat2),
        math.sqrt((math.cos(lat1) + bx)**2 + by**2)
    )
    mid_lon = lon1 + math.atan2(by, math.cos(lat1) + bx)

    return [math.degrees(mid_lon), math.degrees(mid_lat)]


# Compare spherical midpoint vs. naive coordinate average
dal = [-96.80, 32.78]
lon_city = [-0.13, 51.51]

mid_spherical = great_circle_midpoint(dal, lon_city)
mid_naive     = [(dal[0] + lon_city[0]) / 2, (dal[1] + lon_city[1]) / 2]

print("Dallas → London midpoints:")
print(f"  Spherical: {mid_spherical[0]:.3f}°E, {mid_spherical[1]:.3f}°N")
print(f"  Naive avg: {mid_naive[0]:.3f}°E, {mid_naive[1]:.3f}°N")
print()

# How far apart are the two midpoints?
separation = haversine_km(mid_spherical, mid_naive)
print(f"Separation between the two midpoints: {separation:.1f} km")
print()

# Bearing at the spherical midpoint
mid_bearing = compute_bearing(mid_spherical, lon_city)
init_bearing = compute_bearing(dal, lon_city)
print(f"Initial bearing (Dallas):        {init_bearing:.2f}°")
print(f"Bearing at spherical midpoint:   {mid_bearing:.2f}°")
print(f"Final bearing (arriving London): {final_bearing(dal, lon_city):.2f}°")

The naive midpoint (coordinate average) is offset from the true great-circle midpoint by hundreds of kilometers on an intercontinental route. The three bearing values — initial, midpoint, final — trace the heading evolution across the arc.

## 4. Building a Great-Circle Path

Sampling a great-circle arc as a sequence of points lets you draw it accurately on a map and analyze the bearing at any position along the route. The approach: step from p1 to p2 in `n` increments using `destination_point` along the correct bearing at each step, or more simply by interpolating the arc fraction using the spherical linear interpolation formula (slerp).

A clean approximation: interpolate `n+1` fractions from `0` to `1`, and at each fraction `t` compute the intermediate point by stepping `t × total_distance` from `p1` at the initial bearing.

In [ ]:
def great_circle_path(p1, p2, n=50):
    """
    Sample n+1 points along the great-circle arc from p1 to p2.
    Returns a list of [lon, lat] points including both endpoints.
    """
    total_dist = haversine_km(p1, p2)
    initial_brg = compute_bearing(p1, p2)
    points = []
    for i in range(n + 1):
        t = i / n
        pt = destination_point(p1, initial_brg, t * total_dist)
        # Recompute bearing from current point for accuracy on long arcs
        if i > 0 and i < n:
            initial_brg = compute_bearing(pt, p2)
        points.append(pt)
    return points


# Build and display the great-circle path Dallas → London
from ipyleaflet import Map, GeoJSON

gc_points = great_circle_path(dal, lon_city, n=40)

gc_line = {
    "type": "Feature",
    "geometry": {"type": "LineString", "coordinates": gc_points},
    "properties": {}
}

# Also build a naive straight-coordinate line for comparison
straight_line = {
    "type": "Feature",
    "geometry": {"type": "LineString", "coordinates": [dal, lon_city]},
    "properties": {}
}

endpoints_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": dal},       "properties": {"name": "Dallas"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": lon_city},  "properties": {"name": "London"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": mid_spherical}, "properties": {"name": "Midpoint"}},
    ]
}

m = Map(center=(45, -50), zoom=3)
m.add(GeoJSON(data={"type":"FeatureCollection","features":[gc_line]},
              style={"color": "#e63946", "weight": 2}))
m.add(GeoJSON(data={"type":"FeatureCollection","features":[straight_line]},
              style={"color": "#aaaaaa", "weight": 1.5, "dashArray": "5"}))
m.add(GeoJSON(data=endpoints_fc,
              point_style={"radius": 5, "color": "#457b9d", "fillOpacity": 1.0}))
m

The red arc is the great-circle path — the actual shortest route. The dashed gray line is the naive straight-coordinate line. They share the same endpoints but diverge significantly in between. The red arc arcs north over the Atlantic; the gray line cuts straight across at constant latitude.

## 5. Bearing to a Moving Target

Static targets have a fixed bearing. Moving targets do not. As a target changes position over time, the required bearing from a fixed launch site changes with it.

This is the core of intercept geometry: the bearing you need to fire is not the bearing to where the target *is* — it is the bearing to where the target *will be* by the time your projectile arrives.

In [ ]:
# A target moving eastward at constant speed from a known starting point
# We track how the required bearing from Wichita Falls changes over time

launch_site    = [-98.49, 33.91]   # Wichita Falls
target_start   = [-101.0, 36.0]    # northwest of WF
target_bearing = 90                # target heading: due east
target_speed   = 250               # km per simulated hour

print(f"Launch site: Wichita Falls")
print(f"Target starts at: {target_start}  heading: {target_bearing}°  speed: {target_speed} km/h")
print()
print(f"{'Time (h)':>10}  {'Target position':>30}  {'Required bearing':>18}  {'Distance':>12}")
print("-" * 78)

for hour in range(0, 6):
    target_pos = destination_point(target_start, target_bearing, hour * target_speed)
    brg  = compute_bearing(launch_site, target_pos)
    dist = haversine_km(launch_site, target_pos)
    print(f"{hour:>9}h  [{target_pos[0]:>8.3f}, {target_pos[1]:>7.3f}]  {brg:>17.2f}°  {dist:>10.1f} km")

The required bearing shifts continuously as the target moves east. At hour 0 the target is northwest; by hour 4 it has moved far enough east that the required bearing is also shifting eastward. This is bearing-over-time — the foundation of lead-angle and intercept calculation.

---

## Exercise A — Multi-Leg Route

A route makes three stops: Wichita Falls → Chicago → New York → London.

For each leg, compute and print:
- initial bearing
- final bearing  
- distance
- the delta between initial and final bearing

Then compute the **total route distance** and identify which leg has the largest bearing drift.

In [ ]:
waypoints = [
    ("Wichita Falls", [-98.49, 33.91]),
    ("Chicago",       [-87.63, 41.88]),
    ("New York",      [-74.01, 40.71]),
    ("London",        [ -0.13, 51.51]),
]

# your code here

## Exercise B — Great-Circle Path on a Map

Using `great_circle_path`, draw the arcs for all three legs of the route above on a single ipyleaflet map. Use a different color per leg. Add markers at each waypoint.

In [ ]:
leg_colors = ["#e63946", "#457b9d", "#2a9d8f"]

# your code here
# hint: zip(waypoints[:-1], waypoints[1:]) to iterate leg pairs

## Exercise C — Intercept Bearing

A target is moving due north at 300 km/h, starting from `[-97.0, 30.0]`. Your launch site is at `[-98.49, 33.91]` (Wichita Falls). Your projectile travels at 600 km/h.

Find the intercept: the target position and required bearing such that your projectile, fired now, arrives at the same point at the same time as the target.

Approach:
1. For each time step `t` (in hours, try 0.1 to 3.0 in steps of 0.1), compute where the target will be
2. Compute the distance from your launch site to that future position
3. Compute the time your projectile would take to cover that distance at 600 km/h
4. Find the `t` where projectile travel time ≈ `t` (they meet)
5. Report the intercept point and the bearing to fire

In [ ]:
target_start   = [-97.0, 30.0]
target_heading = 0      # due north
target_speed   = 300    # km/h
projectile_speed = 600  # km/h
launch = [-98.49, 33.91]

# your code here
# hint: find t where haversine_km(launch, target_at_t) / projectile_speed ≈ t

---

## Check Your Understanding

A student wants to find the midpoint of the Dallas → London route. They write:

```python
mid = [(dal[0] + lon[0]) / 2, (dal[1] + lon[1]) / 2]
```

They verify it is "in the middle" by checking that `haversine_km(dal, mid)` and `haversine_km(mid, lon)` are roughly equal — and they are (both around 3,940 km). So they conclude the midpoint is correct.

**Question:** The distances check out, but the midpoint is still wrong. How is that possible? What does the distance check actually verify, and what does it fail to verify?

```python
# your answer here
```


---

## Next

In [00 — Problem Setup](../08-Intercept_Persuit_Module_Design/00-Problem_Setup.ipynb), we apply bearing and distance together to trajectory geometry — combining both tools into multi-leg route planning and intercept analysis.